# Phase 3b — Paired significance tests on Phase 3 classification results

Only claims get numbers: un-paired 95% CIs of two models overlap by construction
even when the *difference* is highly significant, because both are evaluated on the
**same 950 test cases**. This notebook runs the authoritative tests on the cached
Phase 3 outputs (no GPU):

- **Paired bootstrap Δ AUROC** (5000 case-level resamples, both models on the same
  resample) for the four headline comparisons;
- **Paired bootstrap Δ sensitivity** at the val-fixed spec≥0.90 thresholds, for the
  operating-point claims;
- **McNemar's exact-style χ²** on discordant positive cases at those thresholds;
- Bonferroni screen across the 6 planned tests (α = 0.0083).

Run in the same Kaggle session/working dir as Phase 3 (reads
`/kaggle/working/phase3_classification/*` caches only).

## 0. Load caches (features for labels; ERM + linear-head probabilities)

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve
from IPython.display import display

SEED = 42
N_BOOT = 5000
ALPHA_BONF = 0.05 / 6
ROOT = Path("/kaggle/working/phase3_classification")
need = ["features_test.npz", "features_val.npz",
        "linear_heads_results.json", "erm_resnet34_probs.json"]
miss = [f for f in need if not (ROOT / f).exists()]
assert not miss, ("missing caches %s - run Phase 3 first (through section 8) "
                  "in this working directory" % miss)

zt = np.load(ROOT / "features_test.npz", allow_pickle=True)
zv = np.load(ROOT / "features_val.npz", allow_pickle=True)
y_test = zt["labels"].astype(int)
y_val = zv["labels"].astype(int)
LIN = json.loads((ROOT / "linear_heads_results.json").read_text())
ERM = json.loads((ROOT / "erm_resnet34_probs.json").read_text())

def get_probs(model, split):
    if model == "ERM":
        return np.array(ERM["probs"][split], dtype=float).ravel()
    return np.array(LIN[model]["probs_" + split], dtype=float).ravel()

MODELS = ["ERM", "R0_image", "R1p_lesion", "R2p_lesion_ring", "R1g_gtbox", "R2g_gtbox_ring"]
P_TE = {m: get_probs(m, "test") for m in MODELS}
P_VA = {m: get_probs(m, "val") for m in MODELS}
print("loaded:", {m: P_TE[m].shape for m in MODELS}, "| test n =", len(y_test),
      "| positives =", int(y_test.sum()))

## 1. Statistics — paired Δ bootstrap and spec≥0.90 thresholds (copied from Phase 3)

In [ ]:
def spec90_threshold(y_val, p_val):
    fpr, tpr, thr = roc_curve(y_val, p_val)
    spec = 1 - fpr
    m = spec >= 0.90
    if m.any():
        i = np.where(m)[0][np.argmax(tpr[m])]
    else:
        i = int(np.argmax(spec))
    return float(thr[i]), float(tpr[i]), float(spec[i])

def paired_boot_auroc(y, pa, pb, n=N_BOOT, seed=SEED):
    y = np.asarray(y); rng = np.random.RandomState(seed); N = len(y); d = []
    for _ in range(n):
        idx = rng.randint(0, N, N)
        if len(np.unique(y[idx])) < 2:
            continue
        d.append(roc_auc_score(y[idx], pa[idx]) - roc_auc_score(y[idx], pb[idx]))
    d = np.array(d)
    p = 2 * min((d <= 0).mean(), (d >= 0).mean())
    return d.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5), min(p, 1.0), len(d)

def paired_boot_sens(y, pa, ta, pb, tb, n=N_BOOT, seed=SEED):
    y = np.asarray(y); pos = y == 1
    rng = np.random.RandomState(seed)
    sa = (pa[pos] >= ta).astype(float); sb = (pb[pos] >= tb).astype(float)
    M = len(sa); d = []
    for _ in range(n):
        idx = rng.randint(0, M, M)
        d.append(sa[idx].mean() - sb[idx].mean())
    d = np.array(d)
    p = 2 * min((d <= 0).mean(), (d >= 0).mean())
    return d.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5), min(p, 1.0), len(d)

def mcnemar_chi2(correct_a, correct_b):
    b = int(np.sum(correct_a & ~correct_b))   # a right, b wrong
    c = int(np.sum(~correct_a & correct_b))   # b right, a wrong
    if b + c == 0:
        return None, b, c
    chi2 = (abs(b - c) - 1) ** 2 / (b + c)
    return chi2, b, c

## 2. The six planned tests
AUROC: T1 `R1p − R0` (lesion grounding, RQ3) · T2 `ERM − R1p` (whole-image advantage) ·
T3 `R1g − R1p` (error-propagation cost, RQ4) · T4 `R1p − R2p` (ring null, H6).
Sensitivity @ spec≥0.90: T5 `R1p − ERM` (operating point) · T6 `R1g − R1p` (+ McNemar on T5).

In [ ]:
rows = []
def rec(test, kind, a, b, mean, lo, hi, p, n, extra=""):
    rows.append({"test": test, "metric": kind, "A": a, "B": b,
                 "delta(A-B)": round(mean, 4), "CI95": f"({lo:.4f}..{hi:.4f})",
                 "p_two_sided": round(p, 5), "sig_bonf": "YES" if p < ALPHA_BONF else "no",
                 "n_boot": n, "note": extra})

mean, lo, hi, p, n = paired_boot_auroc(y_test, P_TE["R1p_lesion"], P_TE["R0_image"])
rec("T1", "AUROC", "R1p_lesion", "R0_image", mean, lo, hi, p, n, "RQ3: lesion > image")
mean, lo, hi, p, n = paired_boot_auroc(y_test, P_TE["ERM"], P_TE["R1p_lesion"])
rec("T2", "AUROC", "ERM", "R1p_lesion", mean, lo, hi, p, n, "whole-image fine-tune advantage")
mean, lo, hi, p, n = paired_boot_auroc(y_test, P_TE["R1g_gtbox"], P_TE["R1p_lesion"])
rec("T3", "AUROC", "R1g_gtbox", "R1p_lesion", mean, lo, hi, p, n, "RQ4: oracle - predicted ROI")
mean, lo, hi, p, n = paired_boot_auroc(y_test, P_TE["R1p_lesion"], P_TE["R2p_lesion_ring"])
rec("T4", "AUROC", "R1p_lesion", "R2p_lesion_ring", mean, lo, hi, p, n, "H6: ring penalty (null)")

thr = {m: spec90_threshold(y_val, P_VA[m])[0] for m in MODELS}

per = []
for m in MODELS:
    t_, sv, sp = spec90_threshold(y_val, P_VA[m])
    per.append({"model": m, "thr": round(t_, 4), "val_spec": round(sp, 4),
                "val_sens": round(sv, 4),
                "test_sens@spec90": round(float((P_TE[m][y_test == 1] >= t_).mean()), 4)})
SPEC90 = pd.DataFrame(per)
SPEC90.to_csv(ROOT / "phase3b_spec90_per_model.csv", index=False)
print("per-model spec>=0.90 operating points (raw-raw convention):")
display(SPEC90)

mean, lo, hi, p, n = paired_boot_sens(y_test, P_TE["R1p_lesion"], thr["R1p_lesion"],
                                      P_TE["ERM"], thr["ERM"])
rec("T5", "sens@spec90", "R1p_lesion", "ERM", mean, lo, hi, p, n, "operating point")
mean, lo, hi, p, n = paired_boot_sens(y_test, P_TE["R1g_gtbox"], thr["R1g_gtbox"],
                                      P_TE["R1p_lesion"], thr["R1p_lesion"])
rec("T6", "sens@spec90", "R1g_gtbox", "R1p_lesion", mean, lo, hi, p, n, "oracle vs predicted")

chi2, b, c = mcnemar_chi2((P_TE["R1p_lesion"][y_test == 1] >= thr["R1p_lesion"]),
                          (P_TE["ERM"][y_test == 1] >= thr["ERM"]))
# mcnemar_chi2(a, b): b = a-right & b-wrong ; c = a-wrong & b-right
print(f"McNemar (T5, test positives): R1p right / ERM wrong = {b}, "
      f"ERM right / R1p wrong = {c}, "
      f"chi2 = {chi2 if chi2 is None else round(chi2, 2)} (3.84 ~ p<0.05, 6.63 ~ p<0.01)")

SIG = pd.DataFrame(rows)
SIG.to_csv(ROOT / "phase3b_paired_significance.csv", index=False)
print()
print(f"Bonferroni alpha = {ALPHA_BONF:.4f} across 6 tests")
display(SIG)

## 3. Reading for the paper
- **T1** decides whether "lesion grounding helps" is a claim or a trend.
- **T2** keeps us honest about the ERM fine-tune (expected significant against us on rank).
- **T3** prices the segmentation error (expected small; significance against zero irrelevant —
  the CI *location and width* are the result).
- **T4** tests the ring null (reported either way).
- **T5/T6 + McNemar** carry the operating-point story.
Paste the full printed table back; these numbers become the Results subsection anchors.